# Navigation Pipeline Evaluation

This notebook evaluates the existing pipeline on a configurable subset of the reproducible validation split:

**RGB → trained U-Net semantic prediction → Module 06 terrain costs → Module 06 A* route**

It uses the saved checkpoint only; it does not train or update model weights. It also does not download data or install packages. Results are for a 256×256 pixel grid, not a physical-world map. A path failure is retained as a failure; endpoints are never moved to manufacture success.

The terrain costs are reused unchanged from Module 06: `{0:255, 1:30, 2:80, 3:255, 4:160, 5:130, 6:255, 7:180}`. Bedrock (class 3) and Rock (class 6) each had validation IoU 0, so navigation decisions based on this model are not safety-critical.

## 1. Environment and evaluation configuration

The first 20 samples of the seed-42 validation ordering are used by default. To compare route success across images, the same configurable start and goal pixel are used on every image. If an endpoint is blocked on a particular image, that route is recorded as failed with a reason; there is no alternate endpoint selection.

In [12]:
import csv
import heapq
import json
import math
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from matplotlib.colors import ListedColormap

PROJECT_ROOT = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (PROJECT_ROOT, *PROJECT_ROOT.parents) if (p / 'notebooks').is_dir()), PROJECT_ROOT)
DATASET_ROOT = Path.home() / 'data' / 'BASEPROD_labeled' / 'BASEPROD' / 'ORIGINAL'
RGB_DIR = DATASET_ROOT / 'COLOR_IMGS'
MASK_DIR = DATASET_ROOT / 'MASK_RAWS'
CHECKPOINT_PATH = PROJECT_ROOT / 'outputs' / 'segmentation' / 'best_unet_rgb.pt'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'navigation_evaluation'
VISUALIZATION_DIR = OUTPUT_DIR / 'visualizations'

NUM_EVAL_IMAGES = 20
RANDOM_SEED = 42
IMAGE_SIZE = (256, 256)  # width, height; same PIL resize as Module 06.
NUM_CLASSES = 8
BASE_CHANNELS = 16
START_XY = (5, 5)
GOAL_XY = (250, 250)
BLOCKED_THRESHOLD = 240
ALLOW_DIAGONAL = True
DEVICE = torch.device('cpu')
CLASS_NAMES = ['Void', 'Compact', 'Grass', 'Bedrock', 'Sandy', 'Pebble', 'Rock', 'Vegetation']
CLASS_COLORS = ['#000000', '#eab676', '#5b9f4a', '#8b8000', '#c04000', '#7393b3', '#e5e4e2', '#74d810']

torch.set_num_threads(max(1, min(4, torch.get_num_threads())))
print(f'Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | device: {DEVICE}')
print(f'CUDA available: {torch.cuda.is_available()} (evaluation is explicitly CPU-only)')
print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'RGB directory: {RGB_DIR}')
if not isinstance(NUM_EVAL_IMAGES, int) or NUM_EVAL_IMAGES < 1:
    raise ValueError('NUM_EVAL_IMAGES must be a positive integer.')
if not RGB_DIR.is_dir() or not MASK_DIR.is_dir():
    raise FileNotFoundError(f'Expected the verified RGB and mask folders under {DATASET_ROOT}.')
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'Trained checkpoint not found: {CHECKPOINT_PATH}')

Python: 3.12.7 | PyTorch: 2.14.1+cpu | device: cpu
CUDA available: False (evaluation is explicitly CPU-only)
Checkpoint: C:\Users\Anushka\UA-MTSeg-Rover-Navigation\UA-MTSeg-Rover-Navigation\outputs\segmentation\best_unet_rgb.pt
RGB directory: C:\Users\Anushka\data\BASEPROD_labeled\BASEPROD\ORIGINAL\COLOR_IMGS


## 2. Reconstruct the verified validation samples

Pairing is by the already verified timestamp basename: `COLOR_IMGS/<timestamp>.png` and `MASK_RAWS/<timestamp>_mask.png`. As in Module 06, sorted complete pairs are shuffled with `random.Random(42)`; the first 190 form validation. We evaluate the first `NUM_EVAL_IMAGES` entries in that exact validation order.

In [13]:
rgb_by_key = {path.stem: path for path in RGB_DIR.glob('*.png')}
mask_by_key = {path.stem.removesuffix('_mask'): path for path in MASK_DIR.glob('*_mask.png')}
if not rgb_by_key or not mask_by_key:
    raise FileNotFoundError('No PNG RGB images or *_mask.png files were found in the verified folders.')
if set(rgb_by_key) != set(mask_by_key):
    missing_masks = sorted(set(rgb_by_key) - set(mask_by_key))[:5]
    missing_rgb = sorted(set(mask_by_key) - set(rgb_by_key))[:5]
    raise ValueError(f'RGB/mask pairing mismatch. Missing masks examples={missing_masks}; missing RGB examples={missing_rgb}')
samples = [{'key': key, 'rgb_path': rgb_by_key[key], 'mask_path': mask_by_key[key]} for key in sorted(rgb_by_key)]
if len(samples) != 950:
    raise ValueError(f'Expected the verified 950 pairs, found {len(samples)}; refusing to evaluate a different dataset silently.')
shuffled_samples = list(samples)
random.Random(RANDOM_SEED).shuffle(shuffled_samples)
validation_count = max(1, int(round(0.2 * len(shuffled_samples))))
validation_samples = shuffled_samples[:validation_count]
if NUM_EVAL_IMAGES > len(validation_samples):
    raise ValueError(f'NUM_EVAL_IMAGES={NUM_EVAL_IMAGES} exceeds validation size {len(validation_samples)}.')
evaluation_samples = validation_samples[:NUM_EVAL_IMAGES]
print(f'Complete verified pairs: {len(samples)}')
print(f'Reproducible split: {len(samples) - len(validation_samples)} train / {len(validation_samples)} validation (seed={RANDOM_SEED})')
print(f'Images selected for evaluation: {len(evaluation_samples)}')
print('Configured common start/goal (x, y):', START_XY, '->', GOAL_XY)

Complete verified pairs: 950
Reproducible split: 760 train / 190 validation (seed=42)
Images selected for evaluation: 20
Configured common start/goal (x, y): (5, 5) -> (250, 250)


## 3. Load the existing U-Net checkpoint on CPU

The U-Net definition below matches Module 06. Loading is strict, so a checkpoint/model architecture mismatch raises an error rather than being ignored. No optimizer or training code is used.

In [14]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layers(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=NUM_CLASSES, base_channels=BASE_CHANNELS):
        super().__init__()
        widths = [base_channels, base_channels * 2, base_channels * 4, base_channels * 8]
        self.enc1 = ConvBlock(in_channels, widths[0])
        self.enc2 = ConvBlock(widths[0], widths[1])
        self.enc3 = ConvBlock(widths[1], widths[2])
        self.enc4 = ConvBlock(widths[2], widths[3])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(widths[3], widths[3] * 2)
        self.up4 = nn.ConvTranspose2d(widths[3] * 2, widths[3], kernel_size=2, stride=2)
        self.dec4 = ConvBlock(widths[3] * 2, widths[3])
        self.up3 = nn.ConvTranspose2d(widths[3], widths[2], kernel_size=2, stride=2)
        self.dec3 = ConvBlock(widths[2] * 2, widths[2])
        self.up2 = nn.ConvTranspose2d(widths[2], widths[1], kernel_size=2, stride=2)
        self.dec2 = ConvBlock(widths[1] * 2, widths[1])
        self.up1 = nn.ConvTranspose2d(widths[1], widths[0], kernel_size=2, stride=2)
        self.dec1 = ConvBlock(widths[0] * 2, widths[0])
        self.classifier = nn.Conv2d(widths[0], num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat((self.up4(b), e4), dim=1))
        d3 = self.dec3(torch.cat((self.up3(d4), e3), dim=1))
        d2 = self.dec2(torch.cat((self.up2(d3), e2), dim=1))
        d1 = self.dec1(torch.cat((self.up1(d2), e1), dim=1))
        return self.classifier(d1)


checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
if 'model_state_dict' not in checkpoint:
    raise KeyError(f'Checkpoint is missing model_state_dict: {CHECKPOINT_PATH}')
model = UNet().to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval()
print(f'Checkpoint loaded successfully: {CHECKPOINT_PATH.name}; model is in eval mode on {DEVICE}.')

Checkpoint loaded successfully: best_unet_rgb.pt; model is in eval mode on cpu.


## 4. Reuse Module 06 terrain costs and A* rules

Costs, blocked threshold, diagonal movement, endpoint behavior, average endpoint cost per move, octile/Manhattan heuristic, and diagonal corner-cut prevention match Module 06. A blocked start or goal counts as a failed route; an unreachable goal stays a failed route.

In [15]:
CLASS_COSTS = {0: 255, 1: 30, 2: 80, 3: 255, 4: 160, 5: 130, 6: 255, 7: 180}
if set(CLASS_COSTS) != set(range(NUM_CLASSES)) or any(not 0 <= cost <= 255 for cost in CLASS_COSTS.values()):
    raise ValueError('The Module 06 class-cost mapping must cover IDs 0..7 with costs in 0..255.')
print('Reused class costs:', CLASS_COSTS)


def astar_path(costs, start_xy, goal_xy, blocked_threshold=BLOCKED_THRESHOLD, allow_diagonal=ALLOW_DIAGONAL):
    height, width = costs.shape
    for name, (x, y) in (('start', start_xy), ('goal', goal_xy)):
        if not (0 <= x < width and 0 <= y < height):
            raise ValueError(f'{name} coordinate {(x, y)} is outside map bounds {width}x{height}.')
        if int(costs[y, x]) >= blocked_threshold:
            raise ValueError(f'{name} coordinate {(x, y)} is blocked with cost {int(costs[y, x])}.')
    if start_xy == goal_xy:
        return [start_xy], 0.0

    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if allow_diagonal:
        moves += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    free_costs = costs[costs < blocked_threshold]
    minimum_cost = float(free_costs.min()) if free_costs.size else 0.0

    def heuristic(x, y):
        distance = math.hypot(goal_xy[0] - x, goal_xy[1] - y) if allow_diagonal else abs(goal_xy[0] - x) + abs(goal_xy[1] - y)
        return minimum_cost * distance

    start_x, start_y = start_xy
    frontier = [(heuristic(start_x, start_y), 0.0, start_y, start_x)]
    best_cost = {start_xy: 0.0}
    parent = {}
    closed = set()
    while frontier:
        _, path_cost, y, x = heapq.heappop(frontier)
        current = (x, y)
        if current in closed:
            continue
        if current == goal_xy:
            path = [goal_xy]
            while path[-1] != start_xy:
                path.append(parent[path[-1]])
            path.reverse()
            return path, path_cost
        closed.add(current)
        for dx, dy in moves:
            nx, ny = x + dx, y + dy
            neighbor = (nx, ny)
            if not (0 <= nx < width and 0 <= ny < height):
                continue
            if int(costs[ny, nx]) >= blocked_threshold or neighbor in closed:
                continue
            if dx and dy and (int(costs[y, nx]) >= blocked_threshold or int(costs[ny, x]) >= blocked_threshold):
                continue
            step_distance = math.sqrt(2) if dx and dy else 1.0
            next_cost = path_cost + step_distance * (int(costs[y, x]) + int(costs[ny, nx])) / 2.0
            if next_cost < best_cost.get(neighbor, float('inf')):
                best_cost[neighbor] = next_cost
                parent[neighbor] = current
                priority = next_cost + heuristic(nx, ny)
                heapq.heappush(frontier, (priority, next_cost, ny, nx))
    return None, float('inf')


def validate_path(path, costs, start_xy, goal_xy):
    height, width = costs.shape
    if not path or path[0] != start_xy or path[-1] != goal_xy:
        raise AssertionError('Planner returned a path with incorrect start/goal endpoints.')
    for x, y in path:
        if not (0 <= x < width and 0 <= y < height):
            raise AssertionError(f'Planner returned out-of-bounds pixel {(x, y)}.')
        if int(costs[y, x]) >= BLOCKED_THRESHOLD:
            raise AssertionError(f'Planner path entered a blocked pixel {(x, y)}.')
    for (x0, y0), (x1, y1) in zip(path, path[1:]):
        dx, dy = x1 - x0, y1 - y0
        if max(abs(dx), abs(dy)) != 1 or (dx == 0 and dy == 0):
            raise AssertionError(f'Planner returned a non-adjacent move: {(x0, y0)} -> {(x1, y1)}.')
        if dx and dy and (int(costs[y0, x1]) >= BLOCKED_THRESHOLD or int(costs[y1, x0]) >= BLOCKED_THRESHOLD):
            raise AssertionError(f'Planner path cuts diagonally across a blocked corner at {(x0, y0)}.')

Reused class costs: {0: 255, 1: 30, 2: 80, 3: 255, 4: 160, 5: 130, 6: 255, 7: 180}


## 5. Run the pipeline on the selected validation images

Every prediction must have exactly the expected 256×256 shape and contain only class IDs 0–7. The converted cost map must contain only configured costs. For each route, the notebook validates boundaries, endpoints, blocked cells, and legal neighboring steps. Unexpected errors stop execution instead of being silently skipped.

In [16]:
evaluation_rows = []
representative_results = []
valid_cost_values = set(CLASS_COSTS.values())

for eval_index, sample in enumerate(evaluation_samples):
    with Image.open(sample['rgb_path']) as image:
        original_rgb = image.convert('RGB')
        original_size = original_rgb.size
        resized_rgb = original_rgb.resize(IMAGE_SIZE, resample=Image.Resampling.BILINEAR)
        rgb_array = np.asarray(resized_rgb, dtype=np.uint8)
    with Image.open(sample['mask_path']) as mask_image:
        if len(np.asarray(mask_image).shape) != 2:
            raise ValueError(f'Expected a single-channel paired mask for {sample["key"]}: {sample["mask_path"]}')
        if mask_image.size != original_size:
            raise ValueError(f'RGB/mask dimensions differ for {sample["key"]}: {original_size} vs {mask_image.size}')

    input_tensor = torch.from_numpy(np.ascontiguousarray(rgb_array.transpose(2, 0, 1))).float().div_(255.0).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        logits = model(input_tensor)
        prediction = logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)
    expected_shape = (IMAGE_SIZE[1], IMAGE_SIZE[0])
    if tuple(logits.shape) != (1, NUM_CLASSES, *expected_shape) or prediction.shape != expected_shape:
        raise ValueError(f'Unexpected prediction shape for {sample["key"]}: logits={tuple(logits.shape)}, mask={prediction.shape}')
    predicted_ids = set(int(value) for value in np.unique(prediction))
    if not predicted_ids.issubset(set(range(NUM_CLASSES))):
        raise ValueError(f'Unexpected predicted class IDs for {sample["key"]}: {sorted(predicted_ids)}')

    costs = np.zeros_like(prediction, dtype=np.uint8)
    for class_id, terrain_cost in CLASS_COSTS.items():
        costs[prediction == class_id] = terrain_cost
    observed_costs = set(int(value) for value in np.unique(costs))
    if not observed_costs.issubset(valid_cost_values) or costs.min() < 0 or costs.max() > 255:
        raise ValueError(f'Invalid cost map values for {sample["key"]}: {sorted(observed_costs)}')

    height, width = costs.shape
    for endpoint_name, (x, y) in (('start', START_XY), ('goal', GOAL_XY)):
        if not (0 <= x < width and 0 <= y < height):
            raise ValueError(f'Configured {endpoint_name} {x, y} is outside the {width}x{height} map.')
    failure_reason = ''
    if int(costs[START_XY[1], START_XY[0]]) >= BLOCKED_THRESHOLD:
        path, path_cost = None, float('inf')
        failure_reason = 'blocked_start'
    elif int(costs[GOAL_XY[1], GOAL_XY[0]]) >= BLOCKED_THRESHOLD:
        path, path_cost = None, float('inf')
        failure_reason = 'blocked_goal'
    else:
        path, path_cost = astar_path(costs, START_XY, GOAL_XY, BLOCKED_THRESHOLD, ALLOW_DIAGONAL)
        if path is None:
            failure_reason = 'no_route'
        else:
            validate_path(path, costs, START_XY, GOAL_XY)

    path_found = path is not None
    path_steps = len(path) - 1 if path_found else 0
    path_length = sum(math.hypot(b[0] - a[0], b[1] - a[1]) for a, b in zip(path, path[1:])) if path_found else 0.0
    evaluation_rows.append({
        'image_id': sample['key'],
        'path_found': path_found,
        'start_x': START_XY[0],
        'start_y': START_XY[1],
        'goal_x': GOAL_XY[0],
        'goal_y': GOAL_XY[1],
        'path_steps': path_steps,
        'path_length': path_length if path_found else '',
        'path_cost': path_cost if path_found else '',
        'failure_reason': failure_reason,
    })
    if eval_index < 5:
        representative_results.append({
            'image_id': sample['key'],
            'rgb': rgb_array.copy(),
            'prediction': prediction.copy(),
            'costs': costs.copy(),
            'path': path,
            'path_found': path_found,
            'failure_reason': failure_reason,
        })
    print(f'{eval_index + 1:02d}/{len(evaluation_samples)} {sample["key"]}: path_found={path_found}, steps={path_steps}, length={path_length:.2f}, cost={path_cost if path_found else "n/a"}, reason={failure_reason or "-"}')

if len(evaluation_rows) != NUM_EVAL_IMAGES:
    raise AssertionError(f'Expected exactly {NUM_EVAL_IMAGES} result rows, got {len(evaluation_rows)}.')
if len(representative_results) < min(5, NUM_EVAL_IMAGES):
    raise AssertionError('Fewer than five representative predictions were retained.')
print(f'Completed evaluation for {len(evaluation_rows)} images.')

01/20 168995504168600000: path_found=True, steps=272, length=366.44, cost=11168.220766631946, reason=-
02/20 169002556592400000: path_found=False, steps=0, length=0.00, cost=n/a, reason=blocked_start
03/20 168995461707800000: path_found=True, steps=257, length=353.51, cost=12723.550852521856, reason=-
04/20 168994355628300000: path_found=False, steps=0, length=0.00, cost=n/a, reason=blocked_start
05/20 168993763975400000: path_found=False, steps=0, length=0.00, cost=n/a, reason=blocked_start
06/20 168995566349300000: path_found=False, steps=0, length=0.00, cost=n/a, reason=blocked_start
07/20 168993593632800000: path_found=False, steps=0, length=0.00, cost=n/a, reason=blocked_start
08/20 168986980103000000: path_found=False, steps=0, length=0.00, cost=n/a, reason=blocked_start
09/20 168995476228100000: path_found=False, steps=0, length=0.00, cost=n/a, reason=blocked_start
10/20 168995476428700000: path_found=False, steps=0, length=0.00, cost=n/a, reason=no_route
11/20 16899415276420000

## 6. Save per-image results, JSON summary, and visualizations

Failed routes have blank length/cost fields in the CSV and a reason column; successful-path aggregates exclude failed routes. If there are no successful paths, summary averages and extrema are JSON `null` rather than invented zero-valued measurements. Five examples (or all images if fewer than five are configured) are saved, including routes that failed.

In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT_DIR / 'navigation_evaluation.csv'
csv_columns = ['image_id', 'path_found', 'start_x', 'start_y', 'goal_x', 'goal_y', 'path_steps', 'path_length', 'path_cost', 'failure_reason']
with csv_path.open('w', newline='', encoding='utf-8') as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=csv_columns)
    writer.writeheader()
    writer.writerows(evaluation_rows)

successful_rows = [row for row in evaluation_rows if row['path_found']]
failed_count = len(evaluation_rows) - len(successful_rows)
successful_lengths = [float(row['path_length']) for row in successful_rows]
successful_costs = [float(row['path_cost']) for row in successful_rows]
summary = {
    'images_evaluated': len(evaluation_rows),
    'successful_paths': len(successful_rows),
    'failed_paths': failed_count,
    'path_success_rate': len(successful_rows) / len(evaluation_rows) if evaluation_rows else 0.0,
    'average_path_length_successful': float(np.mean(successful_lengths)) if successful_lengths else None,
    'average_path_cost_successful': float(np.mean(successful_costs)) if successful_costs else None,
    'minimum_path_cost': float(min(successful_costs)) if successful_costs else None,
    'maximum_path_cost': float(max(successful_costs)) if successful_costs else None,
    'num_classes': NUM_CLASSES,
    'image_size_width_height': list(IMAGE_SIZE),
    'start_xy': list(START_XY),
    'goal_xy': list(GOAL_XY),
    'validation_split': f'seed-{RANDOM_SEED} reproducible random 80/20; first {NUM_EVAL_IMAGES} of {len(validation_samples)} validation samples',
    'class_costs': {str(key): value for key, value in CLASS_COSTS.items()},
    'failed_route_reasons': {reason: sum(row['failure_reason'] == reason for row in evaluation_rows) for reason in sorted({row['failure_reason'] for row in evaluation_rows if row['failure_reason']})},
    'checkpoint': str(CHECKPOINT_PATH),
}
summary_path = OUTPUT_DIR / 'navigation_evaluation_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

semantic_cmap = ListedColormap(CLASS_COLORS)
for result in representative_results:
    fig, axes = plt.subplots(1, 4, figsize=(18, 5), constrained_layout=True)
    axes[0].imshow(result['rgb'])
    axes[0].set_title('RGB image')
    axes[1].imshow(result['prediction'], cmap=semantic_cmap, vmin=0, vmax=NUM_CLASSES - 1, interpolation='nearest')
    axes[1].set_title('Predicted semantic mask')
    axes[2].imshow(result['costs'], cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    axes[2].set_title('Traversability cost')
    axes[3].imshow(result['costs'], cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    if result['path_found']:
        points = np.asarray(result['path'])
        axes[3].plot(points[:, 0], points[:, 1], color='deepskyblue', linewidth=1.5, label='A* path')
    axes[3].scatter(*START_XY, color='lime', edgecolors='black', s=55, label='Start', zorder=3)
    axes[3].scatter(*GOAL_XY, color='red', marker='*', edgecolors='black', s=100, label='Goal', zorder=3)
    axes[3].set_title('Planned path' if result['path_found'] else f'No path: {result["failure_reason"]}')
    axes[3].legend(loc='lower right')
    for axis in axes:
        axis.axis('off')
    fig.suptitle(f'Validation image {result["image_id"]}')
    figure_path = VISUALIZATION_DIR / f'{result["image_id"]}_pipeline.png'
    fig.savefig(figure_path, dpi=140, bbox_inches='tight')
    plt.close(fig)

histogram_path = OUTPUT_DIR / 'successful_path_cost_distribution.png'
fig, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
if successful_costs:
    axis.hist(successful_costs, bins=min(10, max(1, len(successful_costs))), color='steelblue', edgecolor='black')
    axis.set_xlabel('Accumulated A* path cost (cost-pixels)')
    axis.set_ylabel('Number of successful paths')
    axis.set_title('Successful path-cost distribution')
else:
    axis.text(0.5, 0.5, 'No successful paths in this evaluation subset', ha='center', va='center', transform=axis.transAxes)
    axis.set_axis_off()
fig.savefig(histogram_path, dpi=150, bbox_inches='tight')
plt.close(fig)

print(f'CSV: {csv_path}')
print(f'Summary JSON: {summary_path}')
print(f'Path-cost distribution: {histogram_path}')
print(f'Pipeline visualizations ({len(representative_results)}): {VISUALIZATION_DIR}')
for path in sorted(VISUALIZATION_DIR.glob('*_pipeline.png')):
    print(f'  {path.name} ({path.stat().st_size:,} bytes)')

CSV: C:\Users\Anushka\UA-MTSeg-Rover-Navigation\UA-MTSeg-Rover-Navigation\outputs\navigation_evaluation\navigation_evaluation.csv
Summary JSON: C:\Users\Anushka\UA-MTSeg-Rover-Navigation\UA-MTSeg-Rover-Navigation\outputs\navigation_evaluation\navigation_evaluation_summary.json
Path-cost distribution: C:\Users\Anushka\UA-MTSeg-Rover-Navigation\UA-MTSeg-Rover-Navigation\outputs\navigation_evaluation\successful_path_cost_distribution.png
Pipeline visualizations (5): C:\Users\Anushka\UA-MTSeg-Rover-Navigation\UA-MTSeg-Rover-Navigation\outputs\navigation_evaluation\visualizations
  168993763975400000_pipeline.png (698,427 bytes)
  168994355628300000_pipeline.png (598,870 bytes)
  168995461707800000_pipeline.png (573,194 bytes)
  168995504168600000_pipeline.png (607,815 bytes)
  169002556592400000_pipeline.png (567,952 bytes)


## 7. Evaluation report and limitations

In [18]:
average_length = summary['average_path_length_successful']
average_cost = summary['average_path_cost_successful']
print('Navigation Pipeline Evaluation')
print('------------------------------')
print(f'Images evaluated: {summary["images_evaluated"]}')
print(f'Successful paths: {summary["successful_paths"]}')
print(f'Failed paths: {summary["failed_paths"]}')
print(f'Path success rate: {summary["path_success_rate"] * 100:.1f}%')
print(f'Average path length: {average_length:.2f}' if average_length is not None else 'Average path length: n/a (no successful paths)')
print(f'Average path cost: {average_cost:.2f}' if average_cost is not None else 'Average path cost: n/a (no successful paths)')
print()
print('This is a pixel-grid segmentation-to-path-planning prototype, not real rover navigation.')
print('Bedrock and Rock currently have validation IoU = 0; resulting navigation decisions are not safety-critical.')
print(f'Outputs: {OUTPUT_DIR}')

Navigation Pipeline Evaluation
------------------------------
Images evaluated: 20
Successful paths: 6
Failed paths: 14
Path success rate: 30.0%
Average path length: 377.45
Average path cost: 19119.72

This is a pixel-grid segmentation-to-path-planning prototype, not real rover navigation.
Bedrock and Rock currently have validation IoU = 0; resulting navigation decisions are not safety-critical.
Outputs: C:\Users\Anushka\UA-MTSeg-Rover-Navigation\UA-MTSeg-Rover-Navigation\outputs\navigation_evaluation
